# Lesson 14 · Eigenvalues and SVD — finding the directions a matrix prefers, the key to the low-rank world

**Learning goals (two threads)**
1. **Thread ①**: verify the eigenvector equation `Av = λv` (the axes whose direction doesn't change)
2. **Thread ②**: use SVD for low-rank approximation, `low_rank_approx(A, k)` (keep only the top k singular values)
3. PCA: the "data version" of the same key — find the direction of greatest variation (tied to ①, not a separate topic)

**Why it matters for Aurora**: SVD is the shared mathematical foundation of audio denoising (truncated low-rank STFT), dimensionality-reduction visualizations, and later low-rank adaptation (e.g. LoRA, L84) — this lesson first teaches you to "throw away the unimportant directions."

← **Previous lesson** · [L13 · Special matrices](L13_special_matrices.ipynb)

> Last lesson covered **special matrices**: orthogonal matrices preserve length, symmetric = mirror, and the energy test for positive-definiteness.
> This lesson explores **eigenvalues and SVD**.

## This lesson's story: matrices have "favorite directions"

Have you ever noticed that some people, when pressure gives them a shove, just keep going in their own natural direction — they don't get "pushed off course"?

Matrices have "favorite directions" like that too: **eigenvectors**.

For a matrix A, an eigenvector v satisfies `Av = λv`
Meaning: after A acts on v, v's **direction doesn't change** — only its length gets stretched by a factor of λ. λ is the eigenvalue.

Why does it matter? PCA looks for "the direction in which the data varies most" — and that is exactly the top eigenvector of the covariance matrix. SVD factors any matrix into three parts, `A = U·S·Vᵀ`, and each singular value σᵢ measures how "important" its direction is.

**This lesson's two tasks** (matching the two threads above):
1. Verify Av = λv (an eigenvector's direction doesn't change)
2. Implement `low_rank_approx(A, k)` (keep only the top k singular values — later, low-rank fine-tuning will pick up this same knife)

## 🤔 Why did engineers invent this?

Plenty of people finish a course and still feel they "never really got SVD." The problem is almost never the math — it's that **nobody first made clear what it's for**. Remember the use first; the derivation will follow naturally later.

- **What happens without it?** A 1000×1000 image is a million numbers; a recommender system's "users × movies" rating table can have hundreds of millions of cells. Storing, sending and computing on them as they are is far too expensive — and much of the information inside is repeated and redundant. Engineers needed a general way to "keep the trunk, throw away the twigs."
- **What real problem does it solve?** SVD sorts **any** matrix by importance and splits it into a stack of directions: `A = σ₁u₁v₁ᵀ + σ₂u₂v₂ᵀ + …`. In real data the first few singular values σ usually hold the vast majority of the information. Keep only the top k and you get a small, nearly identical "low-rank" version — **compression, denoising and dimensionality reduction are all, at heart, this one move**.
- **Where will it show up again?** L17 uses eigendecomposition to judge system stability, **LoRA in L84** fine-tunes large models with two low-rank factors; the Month 4 recommender system and the Month 5 embedding visualizations all use the same key.

> One line to remember: **SVD isn't for "computing," it's for "throwing away" — throwing away the unimportant directions.** The code below just helps you see that sentence.


## Quick detour: what exactly is "rank"?

We're about to say "low-rank" and "rank-k approximation" over and over, but we haven't yet pinned down what the word "rank" itself means. Let's hit pause and lay the foundation here, so things don't get muddier as we go.

An analogy: suppose you have a contact list of 100 people, but all 100 belong to one big family group chat — many rows are really "duplicates" or "can be worked out from other rows" (for example, "Dad's entry" and "Mom's entry" both point to the same home address). The truly "independent" information — what can't be pieced together from other rows — is far less than 100 entries.

A matrix's **rank** is about exactly this: **the largest number of mutually "linearly independent" directions you can find in the matrix** (none is a multiple or combination of the others). Concrete examples:

- The matrix `[[1, 2], [2, 4]]`: row 2 = 2 × row 1, so the two rows really provide information about only "one independent direction" — **rank = 1**.
- The matrix `[[1, 0], [0, 1]]` (the identity matrix): the two rows are unrelated, each pointing in its own independent direction — **rank = 2 (full rank)**.

For today, remember one sentence: **rank = the number of independent directions in a matrix**. Once we get to SVD, we'll see a handier equivalent — **a matrix's rank = the number of its nonzero singular values** — and then the phrase "rank-k approximation" will be completely grounded.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

# A rank-1 example: row 2 is a whole-number multiple of row 1, so it adds no new direction
A_rank1 = np.array([[1., 2.], [2., 4.]])
print('rank of A_rank1 =', np.linalg.matrix_rank(A_rank1))

# A rank-2 (full-rank) example: the two rows are independent
A_full = np.array([[1., 0.], [0., 1.]])
print('rank of A_full  =', np.linalg.matrix_rank(A_full))


## 1. Eigenvectors: Av = λv (the axes whose direction doesn't change)

An intuitive example: the diagonal matrix A = [[2, 0], [0, 3]]
- Axis e₁ = [1, 0]: A @ [1, 0] = [2, 0] = **2** × [1, 0]  → eigenvalue λ₁ = 2
- Axis e₂ = [0, 1]: A @ [0, 1] = [0, 3] = **3** × [0, 1]  → eigenvalue λ₂ = 3

A diagonal matrix's eigenvectors are just the coordinate axes. **A non-diagonal matrix's eigenvectors are its own favorite directions, which usually no longer line up with the original coordinate axes.**

`np.linalg.eig(A)` returns `(eigenvalues, eigenvectors)`:
- `vals[i]` is the i-th eigenvalue
- `vecs[:, i]` is the i-th eigenvector (a column vector)

Check: `A @ vecs[:, i] ≈ vals[i] * vecs[:, i]` (both sides are vectors in the same direction)

(That sentence can be confusing on a first read. Put more precisely: for a non-diagonal matrix they're generally no longer the standard x- and y-axes, but a new set of directions — for a symmetric matrix like the one below, a rotated pair of perpendicular directions. Right below we'll nail this down with a concrete example, and also explain properly where the equation `Av = λv` comes from.)

## 1.0 From "direction doesn't change" to `Av = λv` — where does this equation come from?

Above we said "an eigenvector is an axis whose direction doesn't change" — that's plain English. `Av = λv` is the same thing in math. How exactly do the two sentences match up? Let's go step by step.

**Step 1, recall what you learned at school**: multiply a vector v by a number λ (say v=[1,1], λ=3) and you get λv=[3,3]. How does this new vector relate to the original v? — **They lie on the same line**:
- If λ>0 (say 3), λv points the same way as v, just longer;
- If λ<0 (say -3), λv points the opposite way from v (you could also say its direction "turned 180°"), and its length becomes |λ| times as long.

Either way, v and λv are **collinear** — they haven't left the original line. That's exactly what we mean by "direction doesn't change" (strictly speaking, "direction unchanged or exactly reversed").

**Step 2, look at it backwards**: if we find that when A acts on some vector v the result `Av` happens to equal a multiple of v — that is, there is a number λ with `Av = λv` — then `Av` must be collinear with v, and that is precisely the algebraic way to write "direction doesn't change." That's why we **don't** write `Av ∝ v` here (the proportionality sign doesn't say what the multiple is); instead we write the multiple λ out explicitly, because λ itself is a quantity we care about — it tells us "how much was stretched or squeezed."

**Step 3, where the characteristic equation comes from**: move everything in `Av = λv` to the left-hand side:

```
Av - λv = 0
Av - λ(Iv) = 0        # use the identity matrix I to write λv as λIv, so both terms are "matrix × vector"
(A - λI)v = 0
```

This says: "the matrix `(A - λI)` acting on v gives the zero vector." By definition an eigenvector can't be the zero vector (the zero vector has no "direction" to speak of), so v is nonzero — which means the matrix `(A - λI)` "squashes" a nonzero vector down to 0, i.e. it has a nontrivial null space, and so it must be non-invertible (determinant 0). So we can first solve the equation `det(A - λI) = 0` to find every possible λ, then substitute back to find the matching v — and that's where the name "characteristic equation" comes from.

### What does a non-diagonal matrix's eigenvector look like? — A genuinely non-diagonal example

So far we've only looked at a diagonal matrix, whose eigenvectors happen to be the x- and y-axes — a bit too "convenient," and it can trick you into thinking eigenvectors are always coordinate axes. Now let's switch to a genuinely non-diagonal matrix `A = [[3, 2], [2, 3]]` (a symmetric matrix), see where its eigenvectors actually point, and draw a picture to see directly what "direction doesn't change" looks like.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A non-diagonal matrix: the symmetric matrix [[3,2],[2,3]]
A_sym = np.array([[3.0, 2.0], [2.0, 3.0]])
vals_sym, vecs_sym = np.linalg.eig(A_sym)
print('eigenvalues:', vals_sym)
print('eigenvectors (as columns):\n', vecs_sym)
print('Note: these two directions are no longer [1,0] and [0,1] — they have been "turned" to 45°/135°.')

# Draw the eigenvector directions, and the result of applying the matrix to them
plt.figure(figsize=(4.5, 4.5))
origin = np.zeros(2)
colors = ['tab:blue', 'tab:orange']
for i in range(2):
    v = vecs_sym[:, i]
    Av = A_sym @ v
    plt.quiver(*origin, *v, angles='xy', scale_units='xy', scale=1, color=colors[i], label=f'eigenvector v{i+1}')
    plt.quiver(*origin, *Av, angles='xy', scale_units='xy', scale=1, color=colors[i], alpha=0.35, width=0.015)

plt.xlim(-6, 6); plt.ylim(-6, 6)
plt.axhline(0, color='gray', lw=.5); plt.axvline(0, color='gray', lw=.5)
plt.gca().set_aspect('equal'); plt.grid(True, alpha=.25); plt.legend()
plt.title('Solid arrows = eigenvectors, faint arrows = after applying A (same direction, just longer)')
plt.show()

print('Does A @ v1 match λ1 * v1?', np.allclose(A_sym @ vecs_sym[:, 0], vals_sym[0] * vecs_sym[:, 0]))


### Are eigenvalues always real? — Why even a diagonal matrix comes with `+0.j`

You may already have noticed that (even for a matrix as "well-behaved" as a diagonal one) the output of `np.linalg.eig` looks like `2.+0.j`, with an "imaginary part." Here's the rule behind it:

- `np.linalg.eig` is a **general-purpose** function: whether or not the matrix you feed it is symmetric, it always returns results in a "might be complex" format — the array's data type (dtype) is always `complex128`, so even when every imaginary part comes out 0, it still carries the `+0.j` tag. This is just a cautious choice of data type; it doesn't mean anything is wrong with your matrix.
- But whether the imaginary part is "exactly 0" really does depend on the matrix: if the matrix is **real symmetric** (like `[[3,2],[2,3]]`, equal to its own transpose) or diagonal, one can prove its eigenvalues must be real, so the computed imaginary parts come out exactly 0 (only the container is still a complex type).
- For a **general non-symmetric real matrix**, the eigenvalues can genuinely have nonzero imaginary parts. The classic example is the "rotation matrix" `[[0,-1],[1,0]]` — it turns every vector in the plane by 90°, so there is no real vector whose "direction doesn't change," and its eigenvalues can only be genuine complex numbers (the imaginary unit `i` is exactly the bookkeeping mathematicians use so that even "pure rotation" can have eigenvalues).
- If you know for sure the matrix is symmetric and want a version that returns plain real numbers directly, use `np.linalg.eigh` (the h stands for Hermitian/symmetric). It is optimized specifically for symmetric matrices, and its output is ordinary `float64`.

Below we compare three cases: a symmetric matrix (imaginary part 0, but still a complex container), a rotation matrix (nonzero imaginary part), and the purely real result from `eigh`.

In [ ]:
import numpy as np

A_rot = np.array([[0., -1.], [1., 0.]])  # 90-degree rotation matrix
vals_rot, _ = np.linalg.eig(A_rot)
print('eig eigenvalues of the rotation matrix:', vals_rot, ' <- nonzero imaginary part: genuine complex numbers')

A_sym2 = np.array([[3.0, 2.0], [2.0, 3.0]])
vals_sym2, _ = np.linalg.eig(A_sym2)
print('eig  eigenvalues of the symmetric matrix:', vals_sym2, ' <- dtype is still complex128, but the imaginary part is exactly 0')

vals_sym2_h = np.linalg.eigvalsh(A_sym2)
print('eigh eigenvalues of the symmetric matrix:', vals_sym2_h, ' <- dtype is float64, optimized for symmetric matrices')


## Notation first: the key shapes in this section

The key shape changes in this section: the input matrix A is `(m, n)`, an eigenvector v is `(n,)`, and the singular-value vector S is `(min(m,n),)`.

**Full SVD** (`full_matrices=True`, the default): U is `(m, m)`, Vt is `(n, n)`.

**Thin SVD** (`full_matrices=False`): U is `(m, k)`, Vt is `(k, n)`, where `k = min(m, n)` — it keeps only the columns/rows that pair with the min(m, n) singular values, so the result is more compact; it's the usual choice for low-rank approximation. The SVD demo a little further down (the one with A of shape (2,3)) uses thin SVD, which is why Vt.shape=(2,3) rather than (3,3).

The eigenvalue problem `Av = λv` requires A to be square; SVD works for any matrix.

### Digging deeper: where exactly does thin SVD save? Why do full and thin give the same low-rank approximation?

A very practical question: if full SVD can also produce a correct rank-k approximation (in the end you only take the first k columns/rows anyway), why bother having a separate thin SVD?

The answer: **the two SVDs are mathematically equivalent, and the low-rank approximations they produce are identical**. The only difference is "whether you first compute and store the parts that will be thrown away anyway." Full SVD dutifully computes the complete `U (m×m)` and `Vt (n×n)`, even if you only plan to use the first k columns/rows; thin SVD computes only the `min(m,n)`-dimensional part and skips the useless work. The bigger the matrix, and the more m and n differ, the bigger the gap. `low_rank_approx` works with either (the exercise below uses full SVD and then slices, which is just as correct); for very large matrices you'd usually pick thin, saving both time and memory.

Below we test it on a 4×3 matrix: both do a rank-2 approximation, the results should match exactly, but the two approaches store different numbers of values.

In [ ]:
import numpy as np

A_cmp = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.], [10., 11., 12.]])  # 4x3
m, n = A_cmp.shape

U_full, S_full, Vt_full = np.linalg.svd(A_cmp, full_matrices=True)
U_thin, S_thin, Vt_thin = np.linalg.svd(A_cmp, full_matrices=False)
print(f'Full SVD:  U{U_full.shape}  S{S_full.shape}  Vt{Vt_full.shape}')
print(f'Thin SVD:  U{U_thin.shape}  S{S_thin.shape}  Vt{Vt_thin.shape}')

k = 2
A_k_full = U_full[:, :k] @ np.diag(S_full[:k]) @ Vt_full[:k, :]
A_k_thin = U_thin[:, :k] @ np.diag(S_thin[:k]) @ Vt_thin[:k, :]
print('Do the two rank-2 approximations match?', np.allclose(A_k_full, A_k_thin))

# Storage comparison (the full factorization, before slicing out the first k columns/rows)
full_numbers = U_full.size + S_full.size + Vt_full.size
thin_numbers = U_thin.size + S_thin.size + Vt_thin.size
print(f'numbers stored by full SVD: {full_numbers}')
print(f'numbers stored by thin SVD: {thin_numbers}')


## 1.1 How an eigenvector differs from an ordinary vector

When the same matrix acts on different directions, the result may turn a corner, or it may just get longer. Eigenvectors are the directions that "don't turn."


In [ ]:
import numpy as np

A = np.array([[2.0, 0.0], [0.0, 3.0]])
eigen_v = np.array([1.0, 0.0])
ordinary_v = np.array([1.0, 1.0])

print('A @ eigen_v   =', A @ eigen_v, ' direction unchanged')
print('A @ ordinary_v=', A @ ordinary_v, ' direction changed')


In [ ]:
import numpy as np
A = np.array([[2.0,0.0],[0.0,3.0]])
vals, vecs = np.linalg.eig(A)
print('eigenvalues λ:', vals)          # [2. 3.]
print('eigenvectors (as columns):\n', vecs)  # the x-axis and y-axis

## Hands-on

A diagonal matrix's eigenvectors point exactly along the coordinate axes: check whether the numerical difference between `A @ vecs[:, i]` and `vals[i] * vecs[:, i]` is below `1e-10`, then try a vector that isn't an eigen-direction and confirm its direction really does get turned.

### 📐 A small example: why "low-rank = compression" (count the storage by hand)

Treat a small 8×8 grayscale image as an 8×8 matrix: that's **64 numbers**. After an SVD, if the singular values fall off quickly (almost all real images do), keeping just the largest **k=2** singular values is enough to get the gist. Then all you need to store is:

| What to store | How many |
|---|---|
| 2 columns of `u` (8 numbers each) | 2 × 8 = 16 |
| 2 singular values σ | 2 |
| 2 rows of `vᵀ` (8 numbers each) | 2 × 8 = 16 |
| **Total** | **34** |

64 → 34, saving almost half. The bigger the matrix, the more dramatic it gets: a 1000×1000 image (a million numbers) keeping just the top 10 singular values needs `10 × (1000 + 1000 + 1) ≈ 20,000` numbers — **squeezed to 2% of the original**, with almost no visible difference. This is the same intuition behind JPEG-style image compression and model compression. The cell below runs a real matrix to show you the curve "the bigger k, the smaller the error."


In [ ]:
import numpy as np

# SVD factorization: A = U @ diag(σ) @ V^T
A = np.array([[3., 2., 1.],[2., 3., 1.]], float)
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print(f'A ({A.shape}):'); print(A)
print(f'singular values σ = {np.round(s, 4)}')
print(f'U shape={U.shape}, Vt shape={Vt.shape}')
A_rec = U @ np.diag(s) @ Vt
print(f'reconstruction error = {np.max(np.abs(A - A_rec)):.2e}  (should be close to 0)')


### Quick detour: the "Frobenius error" that's about to appear — what is this ruler?

Very soon we'll use the "Frobenius reconstruction error" to measure "how far the approximation is from the original matrix." Let's explain this ruler first, so `norm(..., 'fro')` doesn't leave you scratching your head.

For an ordinary vector we're used to measuring size by "length" (the L2 norm): square each component, add them up, take the square root. The **Frobenius norm** carries exactly that recipe over to matrices — flatten the matrix into one long list of numbers, and again "square every number, add them all up, take the square root":

`||A||_F = sqrt( Σᵢ Σⱼ (A[i,j])² )`

An example: the Frobenius norm of the matrix `[[3, 4]]` = `sqrt(3² + 4²) = sqrt(25) = 5` — exactly the same calculation as the length of the vector `[3,4]`, just treating the "matrix" as a "flattened long vector."

And the "reconstruction error" `||A - A_k||_F` squares every element of the "difference matrix" `(A - A_k)`, adds them up and takes the square root — it measures "how big the cell-by-cell difference between the original and the approximation is, in total." It boils the matrix's overall deviation down to a single number, which makes it easy to compare approximations for different k.

In [ ]:
import numpy as np

toy = np.array([[3., 4.]])
print('by hand sqrt(3^2+4^2)      =', np.sqrt(3**2 + 4**2))
print('np.linalg.norm(toy, "fro") =', np.linalg.norm(toy, 'fro'))

diff = np.array([[1., 2.], [2., 1.]])
manual = np.sqrt(np.sum(diff ** 2))
print('Frobenius norm by hand      =', manual)
print('np.linalg.norm(diff, "fro") =', np.linalg.norm(diff, 'fro'))


## Code experiment: a preview of rank-k approximation — keep only the top k singular values

The reference demo below does an SVD of a 3×3 matrix, keeps 1, 2 and then 3 singular values, and computes the Frobenius reconstruction error each time: the bigger k, the smaller the error, and rank 1 already captures the most important direction. This is exactly the operation the `low_rank_approx` exercise asks you to implement shortly — once you've finished the TODO, come back and compare.

In [ ]:
# Warning: reference demo (finish the low_rank_approx TODO further down first, then come back to compare)
import numpy as np

# SVD rank-k approximation: keep only the top k singular values (variable names get _ref to keep them apart from the exercise)
A_ref = np.array([[1,2,3],[4,5,6],[7,8,9]], float)
U_ref, s_ref, Vt_ref = np.linalg.svd(A_ref)
for k in [1, 2, 3]:
    Ak_ref = U_ref[:, :k] @ np.diag(s_ref[:k]) @ Vt_ref[:k, :]
    err = np.linalg.norm(A_ref - Ak_ref, 'fro')
    print(f'rank-{k} approximation  Frobenius error={err:.4f}')
print('-> rank 1 captures the most important direction; the error is set by sigma_2.')

### Digging deeper: why are singular values always sorted from largest to smallest? Why is "the top k" the best choice?

**Where the ordering rule comes from**: the formal definition of a singular value σᵢ is the square root of an eigenvalue of the matrix `AᵀA` (so it is necessarily non-negative). When mathematicians **defined** the notation for the SVD, they agreed to list them in descending order, `σ₁ ≥ σ₂ ≥ ... ≥ σᵣ ≥ 0` — this isn't a rule NumPy added on its own, it's the standard way of writing the SVD itself (that way the factorization's "order is uniquely fixed," and the matching columns/rows of U and Vt are put in order along with it). NumPy (and almost every linear algebra library) simply follows this mathematical convention; it isn't an arbitrary choice.

**Why the top k is optimal, not just lucky**: the experiment above only observed "the bigger k, the smaller the error." But the deeper question is — what guarantees that "taking the k largest singular values" is the **best** among all "matrices of rank at most k"? Could there be some odd rank-k matrix that is closer to the original than the truncated SVD?

There is a genuine mathematical theorem here, the **Eckart–Young theorem**: among all matrices of rank at most k, `A_k = U[:, :k] @ diag(S[:k]) @ Vt[:k, :]` (the truncated-SVD recipe) is the **closest** one to the original matrix A in the Frobenius norm, and this minimum error equals exactly the "energy" of the singular values that were thrown away:

`min ||A - A_k||_F = sqrt(σ_{k+1}² + σ_{k+2}² + ...)`

This course doesn't ask you to prove the theorem (the proof needs more linear-algebra tools), but here's an intuition: SVD has already sorted the matrix by "how much energy each direction contributes" (σᵢ), from largest to smallest — throw away the directions with the least energy, and the error you're left with is naturally the smallest. It's like packing a suitcase with limited room: you leave out the clothes you wear least, rather than grabbing one at random — once the singular values are sorted, "the top k" are by nature "the k items you wear most."

## 2. PCA: first find the direction in which the data varies most

PCA can be seen as "finding a new set of coordinate axes": the first axis points along the direction in which the data varies most. SVD will give us a more general factorization tool later on.


### What exactly is the relationship between PCA and SVD?

The heading says "PCA," but the code below calls `np.linalg.svd` — and we've just been talking about eigenvalues and SVD. Are these three the same thing or not?

In short: **PCA is the goal; SVD is one of the most convenient tools for doing PCA.**

PCA's goal is to find "new coordinate axes" for a cloud of data points, with the first axis pointing in the direction where the data is "most spread out" (the technical term is "the direction of greatest variance"). One can prove mathematically: if you first center the data X (subtract each column's own mean, which is `Xc = X - X.mean(axis=0)` in the code), then this "direction of greatest variance" is exactly the **first right singular vector** of the SVD of `Xc` (that is, `Vt[0]`, the direction that pairs with the largest singular value).

If you're more familiar with the "covariance matrix + eigenvectors" version of PCA: the top eigenvector of the covariance matrix `Xc.T @ Xc / (n-1)` and `Vt[0]` from the SVD of `Xc` are in fact the same direction — because the eigenvalues of `Xc.T @ Xc` are exactly the squares of the singular values of `Xc`. In other words, "find the eigenvectors of the covariance matrix" and "do an SVD of the data directly" are two roads to the same answer; the SVD road is more numerically stable, so it's the one engineers usually take.

(A small tip: if you're reading this statically after the fact and not running the code live, you may not see the plot below — the end of the code also prints the principal direction's coordinates and tilt angle, so you can confirm the conclusion from the numbers too.)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

X = np.array([
    [0.0, 0.2], [1.0, 1.1], [2.0, 1.9], [3.0, 3.2], [4.0, 3.9]
])
Xc = X - X.mean(axis=0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
principal = Vt[0]

plt.figure(figsize=(4.5, 4.5))
plt.scatter(X[:,0], X[:,1], label='data')
center = X.mean(axis=0)
line = np.vstack([center - principal*2.5, center + principal*2.5])
plt.plot(line[:,0], line[:,1], label='first principal direction')
plt.axis('equal'); plt.grid(True, alpha=.25); plt.legend(); plt.show()
print('principal direction =', np.round(principal, 3))

# Numeric backup (lets you confirm the conclusion even without the plot)
angle_deg = np.degrees(np.arctan2(principal[1], principal[0]))
print(f'tilt angle of the principal direction ≈ {angle_deg:.1f}° (if you cannot see the plot above, this number alone tells you: the data spreads out roughly along this angle)')


## 3. SVD: any matrix can be split as `A = U · S · Vᵀ` (S holds the singular values, measuring how important each direction is)

In [ ]:
A = np.array([[3.0,1.0,1.0],[1.0,3.0,1.0],[1.0,1.0,3.0]])
U, S, Vt = np.linalg.svd(A)
print('singular values S:', np.round(S, 3))
recon = U @ np.diag(S) @ Vt
print('does U·S·Vᵀ rebuild A?', np.allclose(recon, A))

## 3.1 SVD: splitting a matrix into directions and strengths

`A = U · S · Vᵀ` can be read as three steps:

1. `Vᵀ`: switch to a set of input directions.
2. `S`: stretch or squeeze along those directions.
3. `U`: turn to the output directions.

Low-rank approximation just means keeping only the few largest stretch strengths.


### Digging deeper: why this order in particular? Why must U and Vᵀ be orthogonal matrices?

**What the factorization is trying to do, "in pictures"**: engineers wanted to split any matrix A (however complicated or irregular) into three "understandable" moves — rotate, stretch, rotate again. Written as `A = U · S · Vᵀ` and read from right to left: first `Vᵀ` turns the input vector onto a set of "standard directions," then `S` (a diagonal matrix) simply stretches or squeezes along those directions (changing lengths, not directions), and finally `U` turns the result to the right directions in the output space. The reason for splitting it as "turn – stretch – turn" is that "stretching" is the simplest operation to understand and to compress (it's just a few numbers), while the complicated rotation parts are carried by U and Vᵀ at either end.

**Why U and Vᵀ must be orthogonal matrices**: last lesson (L13) showed that the key property of an orthogonal matrix is that it "preserves lengths and angles" — it only turns directions and doesn't change a vector's size. SVD requires the S in the middle to be the only part "in charge of stretching": if U or Vᵀ weren't orthogonal, they would change lengths as a side effect while turning vectors, and then the information "how much was stretched" would get mixed up between the matrices at the ends and the S in the middle — S could no longer tell us, separately and cleanly, "how important each direction is." Requiring U and Vᵀ to be orthogonal is like laying down a rule: "the two in charge of turning may only turn directions, never sneak in some stretching on the side," leaving the stretching entirely to S.

Below, for a concrete 2×2 matrix, we print how x changes as it passes through the three steps `Vᵀ → S → U`, and compare with computing `A @ x` directly.

In [ ]:
import numpy as np

A_step = np.array([[3.0, 1.0], [1.0, 3.0]])
U, S, Vt = np.linalg.svd(A_step)
x = np.array([1.0, 0.0])

step1 = Vt @ x                  # step 1: switch to a new set of input directions
step2 = S * step1                # step 2: stretch along those directions (S is the diagonal of a diagonal matrix, so elementwise multiplication is enough)
step3 = U @ step2                # step 3: turn to the output directions

print('original x        =', x)
print('step 1 Vt@x       =', step1, '  (switch to a new set of directions)')
print('step 2 S*step1    =', step2, '  (stretch along those directions)')
print('step 3 U@step2    =', step3, '  (turn to the output directions)')
print('direct A@x        =', A_step @ x, '  (should match step 3)')
print('do the three steps equal A@x computed directly?', np.allclose(step3, A_step @ x))


### A concrete example Samuel asked about: for the diagonal matrix A = diag([2, 3]), will U and Vᵀ in its SVD be identity matrices?

A diagonal matrix is already "stretching along the coordinate axes," so it needs no extra rotation — if the numbers on the diagonal are already in descending order, U and Vᵀ are exactly identity matrices (or differ only by a sign). But if the diagonal isn't in descending order, the SVD keeps its rule that "σ must be descending" by having U (or Vᵀ) "shuffle" the order — and then U is no longer the identity. The two examples below compare these cases.

In [ ]:
import numpy as np

# Case 1: the diagonal is already descending -> U and Vt should be identity matrices (or differ by a sign)
A_diag = np.diag([3.0, 2.0])
U_d, S_d, Vt_d = np.linalg.svd(A_diag)
print('A = diag([3,2]) (already descending)')
print('U =\n', np.round(U_d, 3))
print('S =', S_d)
print('Vt =\n', np.round(Vt_d, 3))

print()
# Case 2: the diagonal is not descending (2 < 5) -> SVD has to reorder, so U is no longer the identity
A_diag2 = np.diag([2.0, 5.0])
U2, S2, Vt2 = np.linalg.svd(A_diag2)
print('A = diag([2,5]) (not descending)')
print('S =', S2, ' <- still comes out descending automatically!')
print('U =\n', np.round(U2, 3), ' <- no longer the identity, because SVD has to move 5 to the front')


## 4. Exercise: low-rank approximation (keep only the k largest singular values)

This is the core of "compressing a matrix" in LoRA and in recommender systems.

**Hint (structural pseudocode, no exact indexing given)**:
```
1. Do an SVD of A -> get U, S, Vt
2. Take only the first k columns of U -> U_k  (shape: m x k)
3. Take only the first k values of S -> S_k  (shape: k,), turn it into a diagonal matrix with np.diag
4. Take only the first k rows of Vt -> Vt_k (shape: k x n)
5. Multiply the three in order, and return an approximation with the same shape as A
```

### Three things to pin down before writing `low_rank_approx`

- Input: a matrix `A` (shape `(m, n)`) and the rank to keep, `k`
- Key steps: do an SVD of `A`, take the first k columns of U, the first k singular values of S (turned into a diagonal matrix with `np.diag`), and the first k rows of Vt, then multiply the three matrices U_k, S_k_diag, Vt_k in order
- Returns: a rank-k approximation with the same shape as `A`; the smaller `k`, the larger the error

In [ ]:
def low_rank_approx(A, k):
    """Return the rank-k approximation of A (same shape as A).

    Parameters
    ----------
    A : np.ndarray, shape (m, n)
    k : int, number of singular values to keep, 1 <= k <= min(m, n)
    """
    U, S, Vt = np.linalg.svd(A)
    # TODO: rebuild an approximation of A from the top k singular values
    # Hint: take the first k columns of U, the first k values of S (np.diag makes them a diagonal matrix), the first k rows of Vt, and multiply the three
    raise NotImplementedError("TODO: return the rank-k approximation, with the same shape as A")


In [ ]:
try:
    _test_placeholder = low_rank_approx(np.eye(2), 1)
except (NotImplementedError, TypeError):
    print("low_rank_approx isn't implemented yet — finish its TODO above, then run this cell.")
else:
    # Full tests (only run once it's implemented)
    A = np.array([[3.0,1.0,1.0],[1.0,3.0,1.0],[1.0,1.0,3.0]])
    A1 = low_rank_approx(A, 1)   # use only the single most important direction
    A3 = low_rank_approx(A, 3)   # use them all -> rebuilds A exactly
    print('rank-1 approximation:\n', np.round(A1, 2))
    print('rank-3 should = A:', np.allclose(A3, A))
    assert np.allclose(A3, A), 'k = full rank should rebuild A exactly'
    print('\nPassed: you have built the core operation behind LoRA and recommender systems.')

    # Rectangular matrix (4x3): check the shape holds up and full-rank reconstruction is correct
    B = np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.],[10.,11.,12.]])
    B_k2 = low_rank_approx(B, 2)
    assert B_k2.shape == B.shape, f'low_rank_approx on a rectangular matrix should have shape {B.shape}, got {B_k2.shape}'
    B_full = low_rank_approx(B, min(*B.shape))
    assert np.allclose(B_full, B, atol=1e-10), 'full-rank reconstruction of the rectangular matrix failed'
    print('Rectangular matrix (4x3): low_rank_approx shape and full-rank reconstruction both passed')


**🔗 Aurora connection**: the Month 4 recommender (matrix factorization), Month 5 LoRA fine-tuning (low-rank adaptation), and embedding visualizations — all of them are today's SVD idea.

**🎉 Milestone**: you now hold the linear-algebra tool used most often in Aurora's engineering. The rest of this linear-algebra phase builds on it, starting with linear systems in L15.

## 🎨 Picture: a matrix = a sum of rank-1 matrices (the core image of SVD)

In [ ]:
from laviz import style, mat_times_mat_rank1
style()
mat_times_mat_rank1([[1,2],[3,4]],[[5,6],[7,8]]);

## 🎯 The payoff later

The **low-rank approximation** you've worked out by hand today will turn up again and again, unchanged, in **JPEG-style image compression / PCA dimensionality reduction / embedding visualization / L84 LoRA fine-tuning / model compression** — on the surface they look wildly different, but at the core they're all the same sentence: **keep the top k singular values, throw away the other directions**. Hold on tight to today's key; later you'll find that many "advanced" tricks are just this one wearing different clothes.


In [ ]:
import numpy as np

# Condition number κ(A) = σ_max / σ_min: the larger it is, the less numerically stable
for label, A in [
    ('near-orthogonal', np.array([[1.,0.],[0.,1.]])),
    ('ill-conditioned', np.array([[1.,1.],[1.,1.0001]])),
    ('rank-deficient', np.array([[1.,2.],[2.,4.]])),
]:
    s = np.linalg.svd(A, compute_uv=False)
    kappa = s[0]/s[-1] if s[-1]>1e-12 else float('inf')
    print(f'{label:8s}  σ={np.round(s,4)}  κ={kappa:.2f}')


## Parameter experiment: turn one knob at a time

Change `k` from 1 up to the matrix's full rank, and watch how the low-rank approximation error `||A - Ak||` shrinks as k grows. Then swap A for a rank-1 matrix and confirm that the approximation error is zero at k=1.

## Wrap-up

`np.linalg.eig(A)` returns an array of eigenvalues `vals` and a matrix of eigenvectors `vecs`, and you can check column by column that `A @ vecs[:, i] ≈ vals[i] * vecs[:, i]`. `np.linalg.svd(A, full_matrices=True)` returns `U`, `S`, `Vt`, and `U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]` is the core computation of `low_rank_approx(A, k)`. Aurora's audio-denoising pipeline cuts off the smallest few singular values of the STFT spectrum matrix, pushing down the low-energy background noise. LoRA fine-tuning represents a pretrained weight matrix as the product of two low-rank factors — the same truncated-SVD idea. The next lesson (**L15**, linear systems) uses rank to decide whether a system of equations has a unique solution.

---
⬇️ **Checkpoint**: you've read the wrap-up; now do the whiteboard challenge below before ticking off the self-review.


## ✏️ Whiteboard challenge: eigenvalues and SVD by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: A = [[2, 0], [0, 3]] (a diagonal matrix) — write down its eigenvalues and eigenvectors directly.
(No calculation needed; the rule for a diagonal matrix: λᵢ = the diagonal entries, vᵢ = the matching coordinate axes)

**Question 2**: what shape does `low_rank_approx(A, k)` output?
If A has shape (3, 5) and k=2, is the output shape still (3, 5), or (3, 2)?

**Question 3**: in an SVD, what ordering rule does `S` (the vector of singular values) follow?
Why does the k=1 approximation use the first singular value rather than the last?

**Question 4**: A = np.eye(3) (the 3×3 identity matrix) — what are its singular values?
(Hint: I = U·S·Vᵀ with U=V=I, so S = ?)

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: eigenvalues/eigenvectors of a diagonal matrix
A1 = np.array([[2., 0.], [0., 3.]])
vals, vecs = np.linalg.eig(A1)
assert set(np.round(np.sort(vals), 6)) == {2.0, 3.0}
print(f"Q1 ✅  diag(2,3) eigenvalues={np.sort(vals)}, eigenvectors = the coordinate-axis directions")
# Check Av = λv
for i in range(2):
    Av = A1 @ vecs[:, i]
    lv = vals[i] * vecs[:, i]
    assert np.allclose(Av, lv, atol=1e-10)
print(f"       Av=λv check passed (both columns keep their direction)")

# Q2: low_rank_approx shape
A2 = np.random.default_rng(42).standard_normal((3, 5))
try:
    approx = low_rank_approx(A2, 2)
    assert approx.shape == A2.shape, f"shape should be {A2.shape}, got {approx.shape}"
    print(f"\nQ2 ✅  low_rank_approx(shape=(3,5), k=2) → shape={approx.shape} (same as A)")
except (NotImplementedError, TypeError):
    print("\n⬜ Q2: implement low_rank_approx() first, then run the answer-check cell")

# Q3: ordering of the singular values
U, S, Vt = np.linalg.svd(A2)
assert all(S[i] >= S[i+1] for i in range(len(S)-1)), "S should be in descending order"
print(f"Q3 ✅  singular values in descending order S[0]≥S[1]≥... (the biggest direction matters most; k=1 uses S[0])")

# Q4: singular values of the identity matrix
I3 = np.eye(3)
_, S_I, _ = np.linalg.svd(I3)
assert np.allclose(S_I, np.ones(3), atol=1e-12)
print(f"Q4 ✅  singular values of eye(3) = {S_I} (all 1: every direction of the identity matters equally)")
print("\n🎉 Eigenvalue & SVD whiteboard challenge passed! Direction-preserving axes and low-rank factorization are now yours.")

In [ ]:
# ✏️ Self-review for this lesson
l14_review = {
    "eigenvector_intuition":     None,  # understand Av=λv: an eigenvector keeps its direction? True/False
    "low_rank_approx_implemented": None,  # low_rank_approx implemented and passes its asserts? True/False
    "svd_decomposition_understood": None,  # understand what each part of A=UΣVᵀ means? True/False
    "singular_value_ordering":   None,  # know singular values are descending, and k=1 uses the largest? True/False
    "whiteboard_passed":         None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l14_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l14_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L14 complete! On to L15: Gaussian elimination')

---

→ **Next lesson** · [L15 · Gaussian elimination](L15_linear_systems.ipynb)

> Next lesson covers **Gaussian elimination**: the elimination process for a system Ax=b, row-echelon form, and classifying when solutions exist.